In [10]:
from selenium import webdriver
import time
import requests #fetches data from webpage
from bs4 import BeautifulSoup #extracts data we need
import pandas as pd #to store data as csv or excel format
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.kvb.co.in/interest-rates/resident-nro-deposits/" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Set up the Chrome driver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)
driver.get(URL) # Fetch the HTML page
time.sleep(5)  ## Wait for 5 secs the JavaScript to execute (you can adjust the sleep time if needed)
html = driver.page_source # Get the rendered HTML
driver.quit() # Close the driver

#Actual webscraping code starts here
soup=BeautifulSoup(html,"html.parser")
#print(soup.prettify())
table=soup.find_all("table",class_="table table-bordered")
table1=table[0]
table2=table[1]
rows1=table1.find_all("tr")

print("Today's date =",date1)
print("Interest rate file name = ",file_name)

list_of_interest_rates = []
row_count = 0
for i in rows1[1:]:  #Skip heading
    data=i.find_all("td")
    row=['Karur Vysya Bank']
    row.extend([tr.text for tr in data])
    for i in range(0,3):
            row[i] = ' '.join(str(row[i]).strip().replace('\xa0', '').split())
    if row_count == 19: row[1] = "Tax-saver Fixed Deposit"
    if row_count not in {18, 20}: list_of_interest_rates.append(row)
    if row_count == 17: row[1] = "Above 5 years to 10 years"
    row_count += 1

row_count = 0
while row_count < 8:
        list_of_interest_rates[row_count].extend([list_of_interest_rates[row_count][2],list_of_interest_rates[row_count][3]])
        row_count += 1

rows1=table2.find_all("tr")
for i in rows1[1:]:
    data=i.find_all("td")
    if row_count == 9:
        #element1 = list_of_interest_rates[row_count][2]
        #element2  = list_of_interest_rates[row_count][3]
        element1, element2 = list_of_interest_rates[row_count][2:4]
        list_of_interest_rates[row_count].extend([str((float(element1[0:4]) + 0.40)) + '%', 
                                                  str((float(element1[0:4]) + 0.42)) + '%'])
        print(list_of_interest_rates[row_count])
        row_count += 1
    print("row count = ", row_count)
    list_of_interest_rates[row_count].extend([tr.text for tr in data[1:]])
    print(list_of_interest_rates[row_count])
    row_count += 1

print(list_of_interest_rates)

pd.set_option('display.max_columns', None)
int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
print(int_rate_df)
#int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
print("\nKarur Vysya Bank = Success. Number of rows added = ", len(list_of_interest_rates))

Today's date = 03122024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_03122024.csv
row count =  8
['Karur Vysya Bank', '333 days', '7.40%', '7.61%', '7.80%', '8.03%']
['Karur Vysya Bank', '334 days to less than 1 year', '6.50%', '6.66%', '6.9%', '6.92%']
row count =  10
['Karur Vysya Bank', '1 Year to 443 days', '7.00%', '7.19%', '7.40%', '7.61%']
row count =  11
['Karur Vysya Bank', '444 Days', '7.50%', '7.71%', '8.00%', '8.24%']
row count =  12
['Karur Vysya Bank', '445 Days upto 2 years', '7.00%', '7.19%', '7.40%', '7.61%']
row count =  13
['Karur Vysya Bank', 'Above 2 years upto 759 days', '7.00%', '7.19%', '7.40%', '7.61%']
row count =  14
['Karur Vysya Bank', '760 Days special deposit', '7.60%', '7.82%', '8.10%', '8.35%']
row count =  15
['Karur Vysya Bank', '761 Days up to 3 years', '7.00%', '7.19%', '7.40%', '7.61%']
row count =  16
['Karur Vysya Bank', 'Above 3 years upto 5 years', '7.00%', '7.19%', '7.40%', '7.61%']
row count =  17
['Karur Vysya Bank', 'Abov

IndexError: list index out of range